# 🧱 Retaining Wall Type Recommendation Using an MLP and Cost Optimization

Rank four wall concepts from five site inputs, then transparently balance technical suitability and estimated cost.

> Educational concept-screening simulation only—not structural or geotechnical design.

👉 **Open the interactive companion:** [https://retaining-wall-recommender.streamlit.app](https://retaining-wall-recommender.streamlit.app/?stage=start)

## Complete workflow

Site inputs → educational labels → preprocessing → MLP → ranked suitability → cost optimization → sensitivity review.

## Interactive learning journey

- [The Retaining-Wall Choice](https://retaining-wall-recommender.streamlit.app/?stage=problem) — Engineering Recommender
- [Five Site Inputs](https://retaining-wall-recommender.streamlit.app/?stage=inputs) — Tabular Features
- [Engineering Suitability Rules](https://retaining-wall-recommender.streamlit.app/?stage=labels) — Training Labels
- [Simulated Retaining-Wall Cases](https://retaining-wall-recommender.streamlit.app/?stage=data) — Synthetic Dataset
- [Preparing Mixed Site Data](https://retaining-wall-recommender.streamlit.app/?stage=prepare) — Leakage-Safe Preprocessing
- [Ranking Wall Alternatives](https://retaining-wall-recommender.streamlit.app/?stage=model) — MLP Classifier
- [Checking the Recommender](https://retaining-wall-recommender.streamlit.app/?stage=audit) — Model Audit
- [Suitability Versus Cost](https://retaining-wall-recommender.streamlit.app/?stage=cost) — Transparent Cost Optimization
- [Engineering Decision Review](https://retaining-wall-recommender.streamlit.app/?stage=review) — Sensitivity and Human Approval

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.metrics import classification_report,ConfusionMatrixDisplay
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input,Dense,Dropout
from tensorflow.keras.callbacks import EarlyStopping
SEED=42;np.random.seed(SEED);tf.random.set_seed(SEED)
WALLS=np.array(["Gravity wall","Cantilever RCC wall","Anchored wall","MSE wall"])
NUM=["height_m","width_m"];CAT=["soil","groundwater","budget"];FEATURES=NUM+CAT

---
# 1. The Retaining-Wall Choice
### Phase 1 of 5 · Choosing a Wall System

## Part 1 · In civil engineering
A project may use a gravity, cantilever RCC, anchored, or MSE wall.

## Part 2 · The engineering challenge
The most suitable option changes with height, soil, groundwater, available width, and budget.

## Part 3 · Where the AI comes in
Rank the alternatives consistently while keeping the final engineering decision reviewable.

**Civil Engineering:** The Retaining-Wall Choice → **AI:** Engineering Recommender → `four alternatives under competing constraints`

> 🎬 **See this illustrated and interactive:** [https://retaining-wall-recommender.streamlit.app/?stage=problem](https://retaining-wall-recommender.streamlit.app/?stage=problem)

## Part 4 · The technical explanation

This notebook separates two questions: **Which wall concepts fit the site technically?** and **How should illustrative cost influence a technically acceptable shortlist?**

## Part 5 · What you just built

**In the notebook:** Define a multiclass recommendation task plus a separate cost layer.

**Takeaway:** Recommendation narrows alternatives; it does not complete geotechnical or structural design.

[Project overview](https://retaining-wall-recommender.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Five Site Inputs](https://retaining-wall-recommender.streamlit.app/?stage=inputs) ▶

---
# 2. Five Site Inputs
### Phase 1 of 5 · Choosing a Wall System

## Part 1 · In civil engineering
Each case is represented by wall height, soil type, groundwater condition, available width, and budget.

## Part 2 · The engineering challenge
Three features are categorical and site width may rule out otherwise attractive systems.

## Part 3 · Where the AI comes in
Encode categories and preserve the physical meaning of height and available width.

**Civil Engineering:** Five Site Inputs → **AI:** Tabular Features → `height, soil, groundwater, width, budget`

> 🎬 **See this illustrated and interactive:** [https://retaining-wall-recommender.streamlit.app/?stage=inputs](https://retaining-wall-recommender.streamlit.app/?stage=inputs)

## Part 4 · The technical explanation

In [ ]:
example=pd.DataFrame([{"height_m":6.0,"soil":"Sand","groundwater":"Medium","width_m":6.0,"budget":"Medium"}]);example

## Part 5 · What you just built

**In the notebook:** Create and inspect the five required inputs.

**Takeaway:** A concise feature set makes assumptions easier to audit.

◀ [Previous: The Retaining-Wall Choice](https://retaining-wall-recommender.streamlit.app/?stage=problem) &nbsp;|&nbsp; [Project overview](https://retaining-wall-recommender.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Engineering Suitability Rules](https://retaining-wall-recommender.streamlit.app/?stage=labels) ▶

---
# 3. Engineering Suitability Rules
### Phase 2 of 5 · Encoding Engineering Judgement

## Part 1 · In civil engineering
Low walls with space may favour gravity walls; moderate heights may favour cantilever walls; constrained high walls may favour anchors; wide reinforced fills may favour MSE.

## Part 2 · The engineering challenge
These tendencies are not universal design rules and cannot replace checks for stability, drainage, bearing, reinforcement, or constructability.

## Part 3 · Where the AI comes in
Use declared scoring rules to generate overlapping simulated labels and retain every class.

**Civil Engineering:** Engineering Suitability Rules → **AI:** Training Labels → `declared educational tendencies`

> 🎬 **See this illustrated and interactive:** [https://retaining-wall-recommender.streamlit.app/?stage=labels](https://retaining-wall-recommender.streamlit.app/?stage=labels)

## Part 4 · The technical explanation

In [ ]:
def engineering_scores(h,soil,water,w,budget):
 s=np.array([72,70,55,62],float)+np.array([12 if h<=4 else -7*(h-4),12-2.5*abs(h-5.5),4.5*h,2.5*h])+np.array([8 if w>=4 else -18,5 if w>=2.5 else -12,18 if w<3 else 2,16 if w>=6 else -16])
 if water=="High":s+=np.array([-14,-8,7,-5])
 elif water=="Low":s+=np.array([5,3,0,4])
 if soil=="Clay":s+=np.array([-5,-2,8,-8])
 elif soil=="Gravel":s+=np.array([7,2,-2,6])
 elif soil=="Sand":s+=np.array([2,4,2,7])
 s+={"Low":np.array([7,-3,-15,5]),"Medium":np.array([1,5,-4,5]),"High":np.array([0,3,8,3])}[budget]
 return s
print(dict(zip(WALLS,engineering_scores(6,"Sand","Medium",6,"Medium"))))

## Part 5 · What you just built

**In the notebook:** Build a transparent label generator for four wall types.

**Takeaway:** A supervised model learns the judgement encoded in its labels.

◀ [Previous: Five Site Inputs](https://retaining-wall-recommender.streamlit.app/?stage=inputs) &nbsp;|&nbsp; [Project overview](https://retaining-wall-recommender.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Simulated Retaining-Wall Cases](https://retaining-wall-recommender.streamlit.app/?stage=data) ▶

---
# 4. Simulated Retaining-Wall Cases
### Phase 2 of 5 · Encoding Engineering Judgement

## Part 1 · In civil engineering
Training examples span varied heights, soils, groundwater, widths, and budget levels.

## Part 2 · The engineering challenge
Perfect deterministic labels would exaggerate model performance and hide uncertainty near design boundaries.

## Part 3 · Where the AI comes in
Add controlled variation and split off unseen test cases.

**Civil Engineering:** Simulated Retaining-Wall Cases → **AI:** Synthetic Dataset → `plausible ranges and overlapping classes`

> 🎬 **See this illustrated and interactive:** [https://retaining-wall-recommender.streamlit.app/?stage=data](https://retaining-wall-recommender.streamlit.app/?stage=data)

## Part 4 · The technical explanation

In [ ]:
rng=np.random.default_rng(SEED);n=6000
df=pd.DataFrame({"height_m":rng.uniform(2,12,n),"soil":rng.choice(["Clay","Sand","Gravel","Mixed"],n),"groundwater":rng.choice(["Low","Medium","High"],n,p=[.35,.4,.25]),"width_m":rng.uniform(1,10,n),"budget":rng.choice(["Low","Medium","High"],n,p=[.3,.45,.25])})
scores=np.vstack([engineering_scores(r.height_m,r.soil,r.groundwater,r.width_m,r.budget) for r in df.itertuples()]);scores+=rng.normal(0,6,scores.shape);df["wall_type"]=WALLS[scores.argmax(1)];print(df.wall_type.value_counts());df.head()

## Part 5 · What you just built

**In the notebook:** Generate the dataset and audit class balance.

**Takeaway:** Synthetic data demonstrates workflow, not field validation.

◀ [Previous: Engineering Suitability Rules](https://retaining-wall-recommender.streamlit.app/?stage=labels) &nbsp;|&nbsp; [Project overview](https://retaining-wall-recommender.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Preparing Mixed Site Data](https://retaining-wall-recommender.streamlit.app/?stage=prepare) ▶

---
# 5. Preparing Mixed Site Data
### Phase 3 of 5 · Learning Suitability

## Part 1 · In civil engineering
Numerical and categorical attributes require different preparation.

## Part 2 · The engineering challenge
Encoding or scaling on the complete dataset leaks test information.

## Part 3 · Where the AI comes in
Fit transformations only on training data using a reusable preprocessing pipeline.

**Civil Engineering:** Preparing Mixed Site Data → **AI:** Leakage-Safe Preprocessing → `one-hot encoding plus standardization`

> 🎬 **See this illustrated and interactive:** [https://retaining-wall-recommender.streamlit.app/?stage=prepare](https://retaining-wall-recommender.streamlit.app/?stage=prepare)

## Part 4 · The technical explanation

In [ ]:
train,temp=train_test_split(df,test_size=.30,stratify=df.wall_type,random_state=SEED);val,test=train_test_split(temp,test_size=.50,stratify=temp.wall_type,random_state=SEED)
prep=ColumnTransformer([("num",StandardScaler(),NUM),("cat",OneHotEncoder(handle_unknown="ignore",sparse_output=False),CAT)]).fit(train[FEATURES])
Xtr,Xva,Xte=prep.transform(train[FEATURES]),prep.transform(val[FEATURES]),prep.transform(test[FEATURES]);lookup={w:i for i,w in enumerate(WALLS)};encode=lambda y:np.array([lookup[v] for v in y]);ytr,yva,yte=encode(train.wall_type),encode(val.wall_type),encode(test.wall_type);print(Xtr.shape,Xva.shape,Xte.shape)

## Part 5 · What you just built

**In the notebook:** One-hot encode categories and scale numerical features.

**Takeaway:** The deployed recommender must reuse the training transformations.

◀ [Previous: Simulated Retaining-Wall Cases](https://retaining-wall-recommender.streamlit.app/?stage=data) &nbsp;|&nbsp; [Project overview](https://retaining-wall-recommender.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Ranking Wall Alternatives](https://retaining-wall-recommender.streamlit.app/?stage=model) ▶

---
# 6. Ranking Wall Alternatives
### Phase 3 of 5 · Learning Suitability

## Part 1 · In civil engineering
Engineers need a ranked shortlist rather than a single unexplained label.

## Part 2 · The engineering challenge
The largest probability can still be weak or sensitive to small input changes.

## Part 3 · Where the AI comes in
Train a compact MLP and retain all four softmax probabilities.

**Civil Engineering:** Ranking Wall Alternatives → **AI:** MLP Classifier → `encoded inputs -> Dense32 -> Dense16 -> Softmax4`

> 🎬 **See this illustrated and interactive:** [https://retaining-wall-recommender.streamlit.app/?stage=model](https://retaining-wall-recommender.streamlit.app/?stage=model)

## Part 4 · The technical explanation

In [ ]:
net=Sequential([Input((Xtr.shape[1],)),Dense(32,activation="relu"),Dropout(.12),Dense(16,activation="relu"),Dense(4,activation="softmax")]);net.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"]);early=EarlyStopping(monitor="val_loss",patience=7,restore_best_weights=True);history=net.fit(Xtr,ytr,validation_data=(Xva,yva),epochs=60,batch_size=64,callbacks=[early],verbose=0);pd.DataFrame(history.history)[["loss","val_loss"]].plot();plt.grid(alpha=.2);plt.show();net.summary()

## Part 5 · What you just built

**In the notebook:** Train the MLP with early stopping and plot learning curves.

**Takeaway:** Probability ranking is useful evidence, not proof of suitability.

◀ [Previous: Preparing Mixed Site Data](https://retaining-wall-recommender.streamlit.app/?stage=prepare) &nbsp;|&nbsp; [Project overview](https://retaining-wall-recommender.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Checking the Recommender](https://retaining-wall-recommender.streamlit.app/?stage=audit) ▶

---
# 7. Checking the Recommender
### Phase 3 of 5 · Learning Suitability

## Part 1 · In civil engineering
Confusion between adjacent alternatives can expose ambiguous site conditions.

## Part 2 · The engineering challenge
Accuracy alone hides which wall types are confused and whether confidence is overstated.

## Part 3 · Where the AI comes in
Inspect held-out predictions, confusion matrix, class metrics, and uncertain cases.

**Civil Engineering:** Checking the Recommender → **AI:** Model Audit → `accuracy, confusion matrix, calibration clues`

> 🎬 **See this illustrated and interactive:** [https://retaining-wall-recommender.streamlit.app/?stage=audit](https://retaining-wall-recommender.streamlit.app/?stage=audit)

## Part 4 · The technical explanation

In [ ]:
pred=net.predict(Xte,verbose=0).argmax(1);print(classification_report(yte,pred,target_names=WALLS));ConfusionMatrixDisplay.from_predictions(yte,pred,display_labels=WALLS,xticks_rotation=25,cmap="Blues");plt.show()
p=net.predict(prep.transform(example),verbose=0)[0];ranking=pd.DataFrame({"Wall":WALLS,"Suitability":p}).sort_values("Suitability",ascending=False);display(ranking.style.format({"Suitability":"{:.1%}"}))

## Part 5 · What you just built

**In the notebook:** Evaluate unseen cases and display a full ranked prediction.

**Takeaway:** Audit class-specific behaviour before using recommendations.

◀ [Previous: Ranking Wall Alternatives](https://retaining-wall-recommender.streamlit.app/?stage=model) &nbsp;|&nbsp; [Project overview](https://retaining-wall-recommender.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Suitability Versus Cost](https://retaining-wall-recommender.streamlit.app/?stage=cost) ▶

---
# 8. Suitability Versus Cost
### Phase 4 of 5 · Balancing Cost

## Part 1 · In civil engineering
A technically strong alternative may be unaffordable, while the cheapest option may be unsuitable.

## Part 2 · The engineering challenge
Cost assumptions differ by location, quantities, access, drainage, and market conditions.

## Part 3 · Where the AI comes in
Estimate illustrative costs, normalize them, and expose alpha and beta rather than hiding the trade-off.

**Civil Engineering:** Suitability Versus Cost → **AI:** Transparent Cost Optimization → `alpha*suitability - beta*normalized cost`

> 🎬 **See this illustrated and interactive:** [https://retaining-wall-recommender.streamlit.app/?stage=cost](https://retaining-wall-recommender.streamlit.app/?stage=cost)

## Part 4 · The technical explanation

In [ ]:
BASE_COST={"Gravity wall":2.6,"Cantilever RCC wall":3.4,"Anchored wall":5.1,"MSE wall":2.9};h=float(example.height_m.iloc[0]);w=float(example.width_m.iloc[0]);costs=np.array([BASE_COST[x]*h*(1+.03*h) for x in WALLS]);costs[3]*=.9 if w>=6 else 1.18;costs[2]*=.92 if w<3 else 1.08
alpha=.75;norm=(costs-costs.min())/(costs.max()-costs.min());combined=alpha*p-(1-alpha)*norm;decision=pd.DataFrame({"Wall":WALLS,"Suitability":p,"Estimated_cost_lakh":costs,"Combined_score":combined}).sort_values("Combined_score",ascending=False);display(decision.style.format({"Suitability":"{:.1%}","Estimated_cost_lakh":"₹{:.1f}","Combined_score":"{:.3f}"}));print("Recommended concept:",decision.iloc[0].Wall)

## Part 5 · What you just built

**In the notebook:** Combine predicted suitability with estimated cost and rank alternatives.

**Takeaway:** Cost modifies a technically screened shortlist; it must not rescue an unsafe option.

◀ [Previous: Checking the Recommender](https://retaining-wall-recommender.streamlit.app/?stage=audit) &nbsp;|&nbsp; [Project overview](https://retaining-wall-recommender.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Engineering Decision Review](https://retaining-wall-recommender.streamlit.app/?stage=review) ▶

---
# 9. Engineering Decision Review
### Phase 5 of 5 · Reviewing the Recommendation

## Part 1 · In civil engineering
Final selection requires surveys, soil investigation, calculations, drainage design, codes, constructability, lifecycle cost, and professional approval.

## Part 2 · The engineering challenge
A recommendation can change near category boundaries or under different cost weights.

## Part 3 · Where the AI comes in
Run sensitivity scenarios, show reasons and limitations, and require an engineer to approve or reject the shortlist.

**Civil Engineering:** Engineering Decision Review → **AI:** Sensitivity and Human Approval → `input scenarios, ranking stability, limitations`

> 🎬 **See this illustrated and interactive:** [https://retaining-wall-recommender.streamlit.app/?stage=review](https://retaining-wall-recommender.streamlit.app/?stage=review)

## Part 4 · The technical explanation

In [ ]:
scenarios=pd.DataFrame([{"height_m":h,"soil":"Sand","groundwater":"Medium","width_m":6.0,"budget":"Medium"} for h in [4,6,8,10]]);probs=net.predict(prep.transform(scenarios),verbose=0);summary=scenarios.copy();summary["Top recommendation"]=WALLS[probs.argmax(1)];summary["Confidence"]=probs.max(1);display(summary)
print("Required next steps: survey, ground investigation, stability and bearing checks, structural design, drainage, seismic/code checks, constructability, lifecycle cost, and licensed professional approval.")

## Part 5 · What you just built

**In the notebook:** Compare scenarios and produce the final educational decision card.

**Takeaway:** The AI is a concept-screening assistant, not a retaining-wall designer.

◀ [Previous: Suitability Versus Cost](https://retaining-wall-recommender.streamlit.app/?stage=cost) &nbsp;|&nbsp; [Project overview](https://retaining-wall-recommender.streamlit.app/?stage=start)

---
# Final engineering conclusion

The MLP produces a ranked concept shortlist from five site inputs. A separate, declared cost layer demonstrates trade-offs without hiding technical suitability. Final selection still requires project-specific geotechnical and structural design.